In [0]:
CREATE OR REPLACE TABLE pondolens.dev_gold.dim_location
USING DELTA
AS

WITH all_locations AS (

    -- GRDP locations
    SELECT
        UPPER(TRIM(region_code)) AS location_code,
        TRIM(region_name) AS location,
        LOWER(TRIM(geographic_level)) AS geographic_level
    FROM pondolens.dev_silver.grdp_per_capita
    WHERE region_name IS NOT NULL

    UNION ALL

    -- Unemployment locations
    SELECT
        UPPER(TRIM(region_code)) AS location_code,
        TRIM(region_name) AS location,
        'region' AS geographic_level
    FROM pondolens.dev_silver.unemployment_rate
    WHERE region_name IS NOT NULL

    UNION ALL

    -- Central Office and Nationwide from appropriations
    SELECT
        UPPER(TRIM(region_code)) AS location_code,
        TRIM(region_name) AS location,
        CASE
            WHEN UPPER(TRIM(region_code)) = 'CO'
                THEN 'office'
            WHEN UPPER(TRIM(region_code)) = 'NW'
                THEN 'national'
        END AS geographic_level
    FROM pondolens.dev_silver.regional_appropriations
    WHERE UPPER(TRIM(region_code)) IN ('CO', 'NW')
),

standardized_locations AS (
    SELECT
        CASE
            WHEN UPPER(TRIM(location)) = 'PHILIPPINES'
                THEN 'PHL'
            WHEN UPPER(TRIM(location)) = 'CENTRAL OFFICE'
                THEN 'CO'
            WHEN UPPER(TRIM(location)) = 'NATIONWIDE'
                THEN 'NW'
            ELSE REGEXP_REPLACE(
                UPPER(TRIM(location_code)),
                '[^A-Z0-9]',
                ''
            )
        END AS location_code,

        CASE
            WHEN UPPER(TRIM(location)) = 'PHILIPPINES'
                THEN 'Philippines'
            WHEN UPPER(TRIM(location)) = 'CENTRAL OFFICE'
                THEN 'Central Office'
            WHEN UPPER(TRIM(location)) = 'NATIONWIDE'
                THEN 'Nationwide'
            ELSE TRIM(location)
        END AS location,

        CASE
            WHEN UPPER(TRIM(location_code)) = 'CO'
                THEN 'office'
            WHEN UPPER(TRIM(location_code)) = 'NW'
                THEN 'national'
            WHEN UPPER(TRIM(location)) = 'PHILIPPINES'
                THEN 'national'
            ELSE geographic_level
        END AS geographic_level

    FROM all_locations
),

-- Deduplicate locations
deduplicated_locations AS (
    SELECT
        location_code,
        location,
        geographic_level
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY location_code, geographic_level
                ORDER BY location
            ) AS rn
        FROM standardized_locations
        WHERE location_code IS NOT NULL
          AND location_code <> ''
          AND geographic_level IS NOT NULL
    )
    WHERE rn = 1
),

-- Deduplicate regional boundaries
region_geometry AS (
    SELECT
        location_code,
        psgc,
        geometry_json
    FROM (
        SELECT
            REGEXP_REPLACE(
                UPPER(TRIM(region_code)),
                '[^A-Z0-9]',
                ''
            ) AS location_code,

            adm1_psgc AS psgc,
            geometry_json,

            ROW_NUMBER() OVER (
                PARTITION BY REGEXP_REPLACE(
                    UPPER(TRIM(region_code)),
                    '[^A-Z0-9]',
                    ''
                )
                ORDER BY adm1_psgc
            ) AS rn

        FROM pondolens.dev_silver.region_boundaries
    )
    WHERE rn = 1
),

locations_with_geometry AS (
    SELECT
        l.location_code,
        l.location,
        l.geographic_level,

        CASE
            WHEN l.geographic_level = 'region'
                THEN g.geometry_json
            ELSE NULL
        END AS geometry_json,

        CASE
            WHEN l.geographic_level = 'region'
                THEN g.psgc
            ELSE NULL
        END AS psgc

    FROM deduplicated_locations l

    LEFT JOIN region_geometry g
        ON l.location_code = g.location_code
        AND l.geographic_level = 'region'
)

SELECT
    10000 + ROW_NUMBER() OVER (
        ORDER BY geographic_level, location_code
    ) AS location_id,

    location_code,
    location,
    geographic_level,
    geometry_json,
    psgc

FROM locations_with_geometry;